In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
import joblib
import streamlit as st




In [2]:
df = pd.read_csv("car_price_dataset.csv", sep=";")
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [3]:
# remove missing values
df = df.dropna()

# separate x and Y
X = df.drop(columns=["Price"])
y = df["Price"]

In [4]:
## 
X = pd.get_dummies(X, drop_first=True)

In [5]:
## split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
## Train the model
lr = LinearRegression()
lr.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [7]:
## Check the RMSE for the model

y_pred = lr.predict(X_test)

rmse_lr = root_mean_squared_error(y_test, y_pred)
print("RMSE:", rmse_lr)

RMSE: 64.91473251975509


In [8]:
## Doing the same as above with decision tree model but worse RMSE
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
rmse_tree = root_mean_squared_error(y_test, y_pred_tree)

print("Decision Tree RMSE:", rmse_tree)

Decision Tree RMSE: 911.5476597523577


In [9]:
## Same as above but with the ridge regression model. 
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)

y_pred_ridge = ridge.predict(X_test)

rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
print("Ridge RMSE:", rmse_ridge)

Ridge RMSE: 64.96044349594392


In [10]:
## Printing out all three RMSE 
print("Linear RMSE:", rmse_lr)
print("Decision Tree RMSE:", rmse_tree)
print("Ridge RMSE:", rmse_ridge)

Linear RMSE: 64.91473251975509
Decision Tree RMSE: 911.5476597523577
Ridge RMSE: 64.96044349594392


In [11]:
## Since Linear Regression works the best we train it on ALL data and not 80% since we've already used the test data to see which model perfoms the best on it
best_model = LinearRegression()
best_model.fit(X, y)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [15]:
joblib.dump(best_model, "car_price_model.pkl")

['car_price_model.pkl']

In [ ]:
## Code for streamlit
## Load the modell
model = joblib.load("car_price_model.pkl")

st.title("Car Price Predictor")
st.write("Enter car details to estimate the price:")

# Numerical inputs
year = st.number_input("Year", min_value=1990, max_value=2025, value=2020)
engine_size = st.number_input("Engine Size", value=2.0)
mileage = st.number_input("Mileage", value=50000)
doors = st.number_input("Doors", value=4)
owner_count = st.number_input("Owner Count", value=1)

# Categorical inputs
brand = st.selectbox("Brand", [
    "BMW","Chevrolet","Ford","Honda","Hyundai","Kia","Mercedes","Toyota","Volkswagen"
])

model_name = st.selectbox("Model", [
    "5 Series","A3","A4","Accord","C-Class","CR-V","Camry","Civic",
    "Corolla","E-Class","Elantra","Equinox","Explorer","Fiesta","Focus",
    "GLA","Golf","Impala","Malibu","Optima","Passat","Q5","RAV4",
    "Rio","Sonata","Sportage","Tiguan","Tucson","X5"
])

fuel = st.selectbox("Fuel Type", ["Electric","Hybrid","Petrol"])
transmission = st.selectbox("Transmission", ["Manual","Semi-Automatic"])

# Create input dataframe (ALL columns!)
input_data = pd.DataFrame(0, index=[0], columns=model.feature_names_in_)

# Numerical values
input_data.at[0, "Year"] = year
input_data.at[0, "Engine_Size"] = engine_size
input_data.at[0, "Mileage"] = mileage
input_data.at[0, "Doors"] = doors
input_data.at[0, "Owner_Count"] = owner_count

# Dummy variables
input_data.at[0, f"Brand_{brand}"] = 1
input_data.at[0, f"Model_{model_name}"] = 1
input_data.at[0, f"Fuel_Type_{fuel}"] = 1
input_data.at[0, f"Transmission_{transmission}"] = 1

# Predict
if st.button("Predict Price"):
    prediction = model.predict(input_data)
    st.success(f"Estimated Price: ${prediction[0]:,.2f}")


In [18]:
## X.columns To see all kategorisk data

In [ ]:
## Det du gjort i denna uppgift, hade det kunnat användas i verkligheten? Till vad i sådana fall?
# Ja, arbetet i denna uppgift hade kunnat användas i verkligheten för att uppskatta bilpriser baserat på olika egenskaper.
# En sån här modell kan användas av bilhandlare, köpare eller plattformar för att få en snabb värdering av fordon.
# däremot så hade såklart modellen behövts tränas på större och mer representativ data med kontinuerlig träning på modellen